In [28]:
import hashlib
BASE58_ALPHABET = "123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz"


In [29]:
def _double_sha256(data: bytes) -> bytes:
    return hashlib.sha256(hashlib.sha256(data).digest()).digest()

In [30]:
def b58encode(raw_bytes: bytes) ->str:
    leading_zeros = 0
    for byte in raw_bytes:
        if byte == 0:
            leading_zeros += 1
        else:
            break
    
    num = int.from_bytes(raw_bytes, byteorder="big")

    result = []
    while num > 0:
        num, remainder = divmod(num,58)
        result.append(BASE58_ALPHABET[remainder])

    encoded = "".join(reversed(result))

    return ("1" * leading_zeros) + encoded

In [39]:
def b58encode(base58_str: str) -> bytes:   
    leading_ones = 0
    for char in base58_str:
        if char == '1':
            leading_ones += 1
        else:
            break
    
    num = 0
    for char in base58_str:
        if char not in BASE58_ALPHABET:
            raise ValueError(f"Invalid Character '{char}' in Base58 string")
        num = num * 58 + BASE58_ALPHABET.index(char)

    num_bytes = num.to_bytes((num.bit_length() + 7) // 8, byteorder="big") if num > 0 else b""

    return (b"\x00" * leading_ones) + num_bytes
 

        

In [40]:
def b58check_encode(version: bytes, playload: bytes) -> str:
    

    data = version + playload

    checksum = _double_sha256(data)[:4]
    return b58encode(data + checksum)

In [41]:
def b58check_decode(base58_str: str) -> tuple[bytes, bytes]:
    raw_data = b58encode(base58_str)

    if len(raw_data) < 5:
        raise ValueError("Decoded data is too short to contain a valid checksum")
    
    data = raw_data[:-4]
    provided_checksum = raw_data[-4:]

    expected_checksum = _double_sha256(data)[:4]

    if provided_checksum != expected_checksum:
        raise ValueError("Invalid Base58Check checksum! data is corrupt or mistyped.")
    version = data[:1]
    payload = data[1:]
    return version, payload
    

In [42]:
if __name__ == "__main__":

    version_byte = b"\x00"
    
    mock_hash160 = bytes.fromhex("f54a5851e9372b87810a8e60cdd2e7cfd80b6e31")
    
    address = b58check_encode(version_byte, mock_hash160)
    print(f"Encoded Address: {address}")

    try:
        ver, payload = b58check_decode(address)
        print(f"Validation Result: Success!")
        print(f"Version Byte: 0x{ver.hex()}")
        print(f"Payload: {payload.hex()}")
    except ValueError as e:
        print(f"Validation Failed: {e}")
    
    corrupted_address = address[:-1] + ("x" if address[-1] != "x" else "y")
    print(f"\nTesting Corrupted Address: {corrupted_address}")
    try:
        b58check_decode(corrupted_address)
    except ValueError as e:
        print(f"Caught expected error: {e}")

TypeError: 'in <string>' requires string as left operand, not int